# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing if it used to get real traffic, hasn't been touched in a while, and is losing ground in search position." Two signals back this: staleness (behind FlyRank's refresh flag) and position-vs-decline (behind the CTR-fix logic).

In [5]:
import duckdb, getpass
import pandas as pd, numpy as np

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# Build prior (Feb) vs current (March) windows to define a REAL decline label
momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar,
               SUM(gsc_clicks) AS clicks_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.client_hash_id, mar.content_hash_id, mar.imp_last, mar.avg_position_mar, mar.clicks_mar,
           feb.imp_prev
    FROM mar
    LEFT JOIN feb ON mar.client_hash_id = feb.client_hash_id
                  AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL
      AND feb.imp_prev >= 100
"""
signal_df = conn.execute(momentum_query).df()
print(signal_df.shape)
signal_df.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(76837, 6)


,client_hash_id,content_hash_id,imp_last,avg_position_mar,clicks_mar,imp_prev
0,client_e547b89c05043229,content_1eea820697c3b95a,315.0,12.723708,0.0,299.0
1,client_e547b89c05043229,content_9abd8b303f805847,14536.0,5.684235,4.0,733.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,387.0,15.227316,0.0,514.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,4697.0,43.049300,5.0,2931.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,1004.0,14.471366,1.0,970.0


In [6]:
content_query = f"""
    SELECT content_hash_id, content_updated_date
    FROM read_parquet('{REL}/dim_content.parquet')
"""
content_df = conn.execute(content_query).df()

signal_df = signal_df.merge(content_df, on="content_hash_id", how="left")
signal_df["days_since_update"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(signal_df["content_updated_date"])).dt.days
signal_df["is_declining"] = (signal_df["imp_last"] < 0.8 * signal_df["imp_prev"]).astype(int)

print(f"Rows: {len(signal_df)} | Decline rate (base rate): {signal_df['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 76837 | Decline rate (base rate): 0.182


In [7]:
signal_df["staleness_bucket"] = pd.cut(
    signal_df["days_since_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=["0-30d", "31-90d", "91-180d", "180d+"]
)
staleness_check = signal_df.groupby("staleness_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print(staleness_check)
print(f"\nBase rate: {signal_df['is_declining'].mean():.3f}")
# Verdict: fill in CONFIRMED / OPPOSITE / MIXED / FALSE based on whether decline_rate
# rises meaningfully across buckets, given n per bucket clears ~50

                      n  decline_rate
staleness_bucket                     
0-30d                77      0.506494
31-90d            16637      0.228407
91-180d              58      0.034483
180d+                27      0.666667

Base rate: 0.182


/tmp/ipykernel_2488/379070412.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_check = signal_df.groupby("staleness_bucket").agg(


In [8]:
signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position_mar"],
    bins=[0, 3, 10, 20, 50, 10000],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
position_check = signal_df.groupby("position_bucket").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
)
print(position_check)

                     n  decline_rate
position_bucket                     
top_3             5872      0.118358
page_1           36289      0.178704
striking         17241      0.211299
page_3_5         15695      0.171010
deep              1590      0.208805


/tmp/ipykernel_2488/891585520.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_check = signal_df.groupby("position_bucket").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
stale = (signal_df["days_since_update"] >= 180).astype(int)
was_visible = (signal_df["imp_prev"] >= 100).astype(int)
losing_ground = (signal_df["avg_position_mar"] > 10).astype(int)

signal_df["action_score"] = stale * was_visible * signal_df["imp_prev"] * (1 + losing_ground)

def reason_code(row):
    if row["days_since_update"] >= 180 and row["avg_position_mar"] > 10:
        return "stale_and_slipping"
    elif row["days_since_update"] >= 180:
        return "stale_but_ranked_ok"
    else:
        return "not_flagged"

signal_df["reason_code"] = signal_df.apply(reason_code, axis=1)
signal_df["action"] = np.where(signal_df["action_score"] > 0, "review_for_refresh", "no_action")

queue = signal_df.sort_values("action_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")

# Precision@K vs base rate — the skill's own recommended check
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in (10, 20, 50):
    p = precision_at_k(queue["action_score"].values, queue["is_declining"].values, k)
    print(f"Precision@{k}: {p:.3f}  (base rate: {signal_df['is_declining'].mean():.3f})")

Wrote 76837 rows to work/outputs/baseline_action_score.csv
Precision@10: 0.600  (base rate: 0.182)
Precision@20: 0.700  (base rate: 0.182)
Precision@50: 0.380  (base rate: 0.182)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
top10 = queue.head(10)[["content_hash_id", "days_since_update", "imp_prev", "avg_position_mar",
                          "action_score", "reason_code", "action", "is_declining"]]
print(top10.to_string(index=False))

         content_hash_id  days_since_update  imp_prev  avg_position_mar  action_score         reason_code             action  is_declining
content_bea86ce3455100b0                232    2749.0          6.555793        2749.0 stale_but_ranked_ok review_for_refresh             0
content_42ce26be1ec6be00                264    2598.0          4.262553        2598.0 stale_but_ranked_ok review_for_refresh             0
content_3af16a3c5dffb146                252    1912.0               NaN        1912.0 stale_but_ranked_ok review_for_refresh             1
content_715dfb7ac57fc2f3                218    1130.0               NaN        1130.0 stale_but_ranked_ok review_for_refresh             1
content_5c9e0961371c7f2d                234     887.0               NaN         887.0 stale_but_ranked_ok review_for_refresh             1
content_84d0bb5aaf292517                280     790.0               NaN         790.0 stale_but_ranked_ok review_for_refresh             1
content_70537a712a8d554e   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Look for the weakest-looking picks in your top 20 by hand
print(queue.head(20)[["content_hash_id", "days_since_update", "imp_prev", "avg_position_mar", "reason_code"]])

# Leakage check: confirm nothing from March itself or beyond leaked into the RULE inputs
print("\nFeatures the rule used: days_since_update, imp_prev (Feb), avg_position_mar")
print("is_declining (label) computed from: imp_last (March) vs imp_prev (Feb) — imp_last never used as a feature.")
print("avg_position_mar is from the SAME month as the label window — flag this as a caveat, not a clean feature.")

                content_hash_id  days_since_update  imp_prev  \
54508  content_bea86ce3455100b0                232    2749.0   
54478  content_42ce26be1ec6be00                264    2598.0   
54492  content_3af16a3c5dffb146                252    1912.0   
34014  content_715dfb7ac57fc2f3                218    1130.0   
33995  content_5c9e0961371c7f2d                234     887.0   
54470  content_84d0bb5aaf292517                280     790.0   
34010  content_70537a712a8d554e                222     543.0   
34002  content_5271624ae98fff86                231     539.0   
54052  content_c126a43258b574c3                231     242.0   
54524  content_454deefc8b3da983                204     450.0   
54497  content_a99d75e3bf98772e                235     398.0   
34022  content_34b8de016805d44d                210     319.0   
54505  content_9dc017d4ef83c0d9                235     276.0   
54507  content_b37c25257ed39a1c                232     266.0   
33981  content_b136fe79d1dff5b3         

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.